# Test-Subset Representativeness Analysis: `test/` vs `test_sub500/`

Compare the same metrics computed on the full test set (`test/`) and on the 500-patch
subset (`test_sub500/`). A representative subset should produce curves that nearly overlap.

Data is loaded from `SAR_DDC_FPGA_all_runs_WandB.csv`. Run `fetch_wandb_runs.py` to regenerate.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ── Paths ─────────────────────────────────────────────────────────────────────
NB_DIR = Path(".")
WANDB_CSV = NB_DIR / "SAR_DDC_FPGA_all_runs_WandB.csv"
PLOTS_DIR = Path("../results/plots/RD-curves")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Color palette (Okabe-Ito) ─────────────────────────────────────────────────
with open(NB_DIR / "plots_colors.json") as fh:
    C = json.load(fh)

_c_relu = C["activations"]["relu"]
_c_gdn = C["activations"]["gdn"]
_c_error = C["metrics"]["error_bars"]

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print("Setup complete.")

## Load & Filter Runs

In [ ]:
runs_df = pd.read_csv(WANDB_CSV, index_col="id")
print(f"Total runs loaded: {len(runs_df)}")
print(runs_df.columns.tolist())

# ── Parse non-scalar columns ──────────────────────────────────────────────────
runs_df["tags"] = runs_df["tags"].apply(lambda v: json.loads(v) if isinstance(v, str) else [])
runs_df["no_output_padding"] = runs_df["no_output_padding"].map(
    {"True": True, "False": False, True: True, False: False}
)

# ── Filtering ─────────────────────────────────────────────────────────────────
ACCEPTED_DATASETS = ["TSXSSCDataModule"]
runs_df = runs_df[runs_df["data_name"].isin(ACCEPTED_DATASETS)].copy()

ACCEPTED_SEEDS = [0, 1, 2, 3, 4, 5, 6]
runs_df = runs_df[runs_df["seed"].isin(ACCEPTED_SEEDS)]

TAGS_TO_REMOVE = ["debug", "crashed", "lr_search"]
for tag in TAGS_TO_REMOVE:
    mask = runs_df["tags"].apply(lambda ts: tag in ts)
    runs_df = runs_df[~mask]

mask_bad_lr = runs_df["architecture"].isin(["ResFP", "FP"]) & (runs_df["lr"] != 5e-4)
runs_df = runs_df[~mask_bad_lr]

mask_gdn1 = runs_df["model_name"].str.contains("gdn1", na=False)
runs_df = runs_df[~mask_gdn1]


# ── Runs that have BOTH test prefixes ─────────────────────────────────────────
def has_both_prefixes(row) -> bool:
    return not pd.isna(row.get("test/bpp")) and not pd.isna(row.get("test_sub500/bpp"))


runs_both = runs_df[runs_df.apply(has_both_prefixes, axis=1)].copy()
print(f"Total runs after filtering: {len(runs_df)}")
print(f"Runs with both test prefixes: {len(runs_both)}")
print(f"Architectures: {runs_both['architecture'].unique().tolist()}")

## Build Statistics

Aggregate over seeds per (architecture, model_name, no_output_padding, λ).
Statistics are built separately for the `test/` and `test_sub500/` prefixes so the two
can be overlaid on the same plot.

In [ ]:
PREFIXES = ["test", "test_sub500"]
METRICS = ["bpp", "psnr_merlin", "ssim_merlin", "epd_merlin"]
STATS = ["mean", "min", "max", "std"]

# Groups — same as ablation notebook
GROUP_COLS = ["architecture", "model_name", "no_output_padding"]


def safe_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return float("nan")


def build_prefix_statistics(df: pd.DataFrame, prefix: str) -> dict[str, pd.DataFrame]:
    """
    Returns {model_key: DataFrame(index=lmbda, columns=[metric_stat, ...])}
    for the given prefix ('test' or 'test_sub500').
    """
    stats: dict[str, pd.DataFrame] = {}
    for group_vals, grp in df.groupby(GROUP_COLS):
        arch, model_name, no_out_pad = group_vals
        activation = model_name.split("_")[1] if "_" in model_name else model_name
        key = f"{arch}_{activation}" + ("_out_pad" if not no_out_pad else "")

        rows = []
        for lmbda, df_lmbda in grp.groupby("lmbda"):
            row = {"lmbda": lmbda}
            for metric in METRICS:
                col = f"{prefix}/{metric}"
                vals = pd.to_numeric(
                    df_lmbda[col] if col in df_lmbda.columns else pd.Series(dtype=float),
                    errors="coerce",
                ).dropna()
                row[f"{metric}_mean"] = vals.mean()
                row[f"{metric}_min"] = vals.min()
                row[f"{metric}_max"] = vals.max()
                row[f"{metric}_std"] = vals.std()
            rows.append(row)

        if rows:
            stats[key] = pd.DataFrame(rows).set_index("lmbda").sort_index()
    return stats


prefix_stats = {p: build_prefix_statistics(runs_both, p) for p in PREFIXES}
print("Keys (test):", sorted(prefix_stats["test"].keys()))

## Side-by-Side RD Curves

Same visual encoding as the ablation notebook:
- **Color** → activation function (ReLU / GDN)
- **Linestyle** → output_padding variant (solid = no padding, dashed = with padding)
- **Marker** → prefix (circle `o` = full `test/`, cross `x` = `test_sub500/`)

In [ ]:
PREFIX_MARKERS = {"test": "o", "test_sub500": "x"}
PREFIX_LABELS = {"test": "full test/", "test_sub500": "test_sub500/"}


def activation_color(key: str) -> str:
    act = key.split("_")[1] if "_" in key else key
    return C["activations"].get(act, "#999999")


def output_pad_linestyle(key: str) -> str:
    return "dashed" if key.endswith("_out_pad") else "solid"


def plot_prefix_comparison(
    prefix_stats: dict[str, dict[str, pd.DataFrame]],
    metric: str,
    ylabel: str,
    title: str,
    save_name: str | None = None,
) -> None:
    fig, ax = plt.subplots(figsize=(6, 4.5))
    keys = sorted(prefix_stats["test"].keys())

    for key in keys:
        color = activation_color(key)
        ls = output_pad_linestyle(key)

        for prefix, marker in PREFIX_MARKERS.items():
            df = prefix_stats[prefix].get(key)
            if df is None or f"{metric}_mean" not in df.columns:
                continue
            bpp = df["bpp_mean"].values
            m = df[f"{metric}_mean"].values
            m_lo = df[f"{metric}_min"].values
            m_hi = df[f"{metric}_max"].values
            ax.plot(bpp, m, color=color, linestyle=ls, marker=marker, markersize=4, linewidth=1.4)
            ax.fill_between(bpp, m_lo, m_hi, color=color, alpha=0.10)

    ax.set_xlabel("BPP")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)

    # ── Legend ────────────────────────────────────────────────────────────────
    act_handles = [
        Line2D([0], [0], color=C["activations"]["relu"], linewidth=2, label="ReLU"),
        Line2D([0], [0], color=C["activations"]["gdn"], linewidth=2, label="GDN"),
    ]
    ls_handles = [
        Line2D([0], [0], color="#555", linestyle="solid", linewidth=2, label="no out_pad"),
        Line2D([0], [0], color="#555", linestyle="dashed", linewidth=2, label="with out_pad"),
    ]
    prefix_handles = [
        Line2D([0], [0], color="#555", marker="o", linestyle="none", markersize=5, label="test/"),
        Line2D(
            [0],
            [0],
            color="#555",
            marker="x",
            linestyle="none",
            markersize=5,
            label="test_sub500/",
        ),
    ]
    ax.legend(handles=act_handles + ls_handles + prefix_handles, fontsize=8, ncol=2, loc="best")
    ax.xaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
    fig.tight_layout()

    if save_name:
        out = PLOTS_DIR / f"{save_name}.pdf"
        fig.savefig(out, bbox_inches="tight")
        print(f"Saved {out}")

    plt.show()


SUBSET_METRICS = [
    ("psnr_merlin", "PSNR vs MERLIN [dB]"),
    ("ssim_merlin", "SSIM vs MERLIN"),
    ("epd_merlin", "EPD vs MERLIN"),
]

for ref, ylabel in SUBSET_METRICS:
    plot_prefix_comparison(
        prefix_stats,
        metric=ref,
        ylabel=ylabel,
        title=f"test/ vs test_sub500/ — {ylabel}",
        save_name=f"subset_comparison_{ref}",
    )

## Difference Table

Mean difference (`test/` − `test_sub500/`) per metric at each λ.
Values close to zero indicate good representativeness.

In [ ]:
diff_rows = []
diff_metrics = ["bpp", "psnr_merlin", "ssim_merlin", "epd_merlin"]

for key in sorted(prefix_stats["test"].keys()):
    df_full = prefix_stats["test"].get(key)
    df_sub = prefix_stats["test_sub500"].get(key)
    if df_full is None or df_sub is None:
        continue
    common_lmbdas = df_full.index.intersection(df_sub.index)
    for lmbda in common_lmbdas:
        row = {"key": key, "lmbda": lmbda}
        for m in diff_metrics:
            full_val = df_full.loc[lmbda, f"{m}_mean"]
            sub_val = df_sub.loc[lmbda, f"{m}_mean"]
            row[f"Δ{m}"] = full_val - sub_val
        diff_rows.append(row)

diff_df = pd.DataFrame(diff_rows).set_index(["key", "lmbda"])
display(diff_df.round(4))

## Scatter Plot: Full vs Sub500

Each point is one run.  Points on the diagonal indicate perfect agreement between
the full test set and the 500-patch subset.

In [ ]:
SCATTER_METRICS = [
    ("bpp", "BPP"),
    # ("psnr_merlin", "PSNR vs MERLIN [dB]"),
    # ("ssim_merlin", "SSIM vs MERLIN"),
    ("epd_merlin", "EPD vs MERLIN"),
]

fig, axes = plt.subplots(1, len(SCATTER_METRICS), figsize=(5 * len(SCATTER_METRICS), 4.5))

for ax, (metric, label) in zip(axes, SCATTER_METRICS):
    col_full = f"test/{metric}"
    col_sub = f"test_sub500/{metric}"

    xs, ys, colors = [], [], []
    for _, row in runs_both.iterrows():
        x = row.get(col_full)
        y = row.get(col_sub)
        if pd.isna(x) or pd.isna(y):
            continue
        act = str(row.get("model_name", "")).split("_")
        act = act[1] if len(act) > 1 else "relu"
        xs.append(safe_float(x))
        ys.append(safe_float(y))
        colors.append(C["activations"].get(act, "#999999"))

    ax.scatter(xs, ys, c=colors, s=12, alpha=0.7)

    # Diagonal
    lims = [min(xs + ys), max(xs + ys)]
    ax.plot(lims, lims, "k--", linewidth=0.8, alpha=0.5)

    ax.set_xlabel(f"test/ {label}")
    ax.set_ylabel(f"test_sub500/ {label}")
    ax.set_title(label, fontsize=10)
    ax.set_aspect("equal", "datalim")

# Shared legend
legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=C["activations"]["relu"],
        markersize=7,
        label="ReLU",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=C["activations"]["gdn"],
        markersize=7,
        label="GDN",
    ),
]
fig.legend(
    handles=legend_handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.04), fontsize=9
)
fig.tight_layout()

out = PLOTS_DIR / "subset_scatter.pdf"
fig.savefig(out, bbox_inches="tight")
print(f"Saved {out}")
plt.show()